# 11:11 · Episode 2 · Wan batch (both free T4 GPUs)

Animates the 16 story shots of Episode 2, two takes each (32 clips), on both T4 GPUs at once. Clips that already exist are skipped, so if the session stops you can just run it again.

**Settings (right-hand panel):** Accelerator **GPU T4 x2**, Internet **On**.
**Best way to run:** **Save Version → Save & Run All (Commit)**. It runs in the background (you can close the tab), then open the saved version → **Output** → download **ep02_clips.zip**.

Expect about 1.5–3 hours in total. The last cell prints a download link as well.

In [ ]:
# 1. Machine check and settings
import os, shutil, subprocess, json, time
print(subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
CACHE = '/kaggle/tmp/hf' if os.path.exists('/kaggle/tmp') else '/tmp/hf'
os.environ['HF_HOME'] = CACHE; os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
for p in ['/kaggle/tmp', '/tmp', '/kaggle/working']:
    if os.path.exists(p): print(p, 'free GB:', round(shutil.disk_usage(p).free / 1e9, 1))
OUT = '/kaggle/working/ep02'; WORK = '/tmp/ep02'
os.makedirs(OUT, exist_ok=True); os.makedirs(WORK + '/frames', exist_ok=True)
NGPU = int(subprocess.run(['bash', '-c', 'nvidia-smi -L | wc -l'], capture_output=True, text=True).stdout.strip() or 1)
print('GPUs:', NGPU)

In [ ]:
# 2. Libraries (about 2 minutes)
!pip install -q -U "diffusers>=0.35" "transformers>=4.49" accelerate ftfy sentencepiece imageio imageio-ffmpeg hf_transfer

In [ ]:
# 3. The shot list and starting frames for Episode 2
import requests
RAW = 'https://raw.githubusercontent.com/PriyankaKondapaneni/siri.ai/claude/anime-rig-test/shorts/eleven/kaggle/ep02/'
jobs = requests.get(RAW + 'jobs.json', timeout=60).json()
for f in sorted(set(j['frame'] for j in jobs)):
    dst = f'{WORK}/{f}'
    if not os.path.exists(dst): open(dst, 'wb').write(requests.get(RAW + f, timeout=60).content)
json.dump(jobs, open(f'{WORK}/jobs.json', 'w'))
done = [j['id'] for j in jobs if os.path.exists(f"{OUT}/{j['id']}.mp4")]
print(len(jobs), 'jobs,', len(done), 'already done')

In [ ]:
%%writefile /tmp/ep02/common.py
MODEL_ID = 'FastVideo/FastWan2.2-TI2V-5B-FullAttn-Diffusers'
WIDTH, HEIGHT, STEPS = 480, 864, 4
NEG = ('static, still picture, scene change, cut, transition, cross-fade, morphing, different person, extra people, '
       'blurred details, low quality, worst quality, JPEG artifacts, deformed, disfigured, extra fingers, poorly drawn hands, '
       'poorly drawn face, fused fingers, watermark, text, subtitles, 3D render, realistic, photorealistic, glossy')

In [ ]:
%%writefile /tmp/ep02/encode.py
# reads every prompt once with the text encoder and saves the results, then exits (freeing all its memory).
# On two T4s the encoder runs in full precision split across both GPUs (fast and exact); otherwise on the CPU.
import json, torch, sys, time
sys.path.insert(0, '/tmp/ep02')
from common import MODEL_ID, NEG
from transformers import AutoTokenizer, UMT5EncoderModel
jobs = json.load(open('/tmp/ep02/jobs.json'))
t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODEL_ID, subfolder='tokenizer')
ngpu = torch.cuda.device_count()
if ngpu >= 2:
    te = UMT5EncoderModel.from_pretrained(MODEL_ID, subfolder='text_encoder', torch_dtype=torch.float32,
                                          device_map='auto', max_memory={0: '14GiB', 1: '14GiB', 'cpu': '20GiB'}).eval()
    dev = 'cuda:0'
else:
    te = UMT5EncoderModel.from_pretrained(MODEL_ID, subfolder='text_encoder', torch_dtype=torch.bfloat16).eval()
    dev = 'cpu'
print(f'text encoder loaded on {"2 GPUs" if dev != "cpu" else "CPU"} in {time.time() - t0:.0f} s', flush=True)
def enc(text, L=512):
    t = tok([text], padding='max_length', max_length=L, truncation=True, add_special_tokens=True, return_attention_mask=True, return_tensors='pt')
    with torch.no_grad(): e = te(t.input_ids.to(dev), t.attention_mask.to(dev)).last_hidden_state.float().cpu()
    n = int(t.attention_mask.sum()); out = torch.zeros_like(e); out[:, :n] = e[:, :n]
    return out.half()
prompts = sorted(set(j['prompt'] for j in jobs))
embs = {'__neg__': enc(NEG)}
for i, p in enumerate(prompts):
    embs[p] = enc(p); print(f'prompt {i + 1} of {len(prompts)} read ({time.time() - t0:.0f} s)', flush=True)
torch.save(embs, '/tmp/ep02/emb.pt')
print('done', flush=True)


In [ ]:
# 4. Read the prompts (a few minutes; skipped if already done). Progress prints below.
if not os.path.exists('/tmp/ep02/emb.pt'):
    p = subprocess.Popen(['python', '/tmp/ep02/encode.py'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-30:]
        if line.startswith(('text encoder', 'prompt ', 'done')): print(line.rstrip(), flush=True)
    p.wait()
    if p.returncode: print(''.join(tail))
print('prompts ready:', os.path.exists('/tmp/ep02/emb.pt'))


In [ ]:
%%writefile /tmp/ep02/worker.py
# one worker per GPU: loads the video model, then works through its share of the jobs
import json, os, sys, time, torch, numpy as np, imageio
sys.path.insert(0, '/tmp/ep02')
from common import MODEL_ID, WIDTH, HEIGHT, STEPS
from PIL import Image
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline, UniPCMultistepScheduler
gpu, ngpu = int(sys.argv[1]), int(sys.argv[2])
OUT, WORK = '/kaggle/working/ep02', '/tmp/ep02'
jobs = [j for i, j in enumerate(json.load(open(f'{WORK}/jobs.json'))) if i % ngpu == gpu]
jobs = [j for j in jobs if not os.path.exists(f"{OUT}/{j['id']}.mp4")]
if not jobs: print(f'gpu{gpu}: nothing to do', flush=True); sys.exit()
# load one at a time so two copies of the model never sit in RAM together
while gpu > 0 and not os.path.exists(f'{WORK}/ready{gpu - 1}'): time.sleep(5)
embs = torch.load(f'{WORK}/emb.pt')
vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder='vae', torch_dtype=torch.float32)
pipe = WanImageToVideoPipeline.from_pretrained(MODEL_ID, vae=vae, text_encoder=None, tokenizer=None, torch_dtype=torch.float16)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=8.0)
pipe.to('cuda')
open(f'{WORK}/ready{gpu}', 'w').write('1')
print(f'gpu{gpu}: model ready, {len(jobs)} jobs', flush=True)
def fit(path):
    im = Image.open(path).convert('RGB'); w, h = im.size; r = WIDTH / HEIGHT
    if w / h > r: nw = int(h * r); im = im.crop(((w - nw) // 2, 0, (w - nw) // 2 + nw, h))
    else: nh = int(w / r); im = im.crop((0, (h - nh) // 2, w, (h - nh) // 2 + nh))
    return im.resize((WIDTH, HEIGHT), Image.LANCZOS)
tiled = False
for j in jobs:
    t0 = time.time()
    kw = dict(image=fit(f"{WORK}/{j['frame']}"), prompt_embeds=embs[j['prompt']].cuda(), negative_prompt_embeds=embs['__neg__'].cuda(),
              height=HEIGHT, width=WIDTH, num_frames=j['frames'], num_inference_steps=STEPS, guidance_scale=0.0, output_type='np',
              generator=torch.Generator(device='cuda').manual_seed(j['seed']))
    for attempt in range(2):
        try:
            with torch.inference_mode(): frames = pipe(**kw).frames[0]
            break
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if tiled: raise
            pipe.vae.enable_tiling(); tiled = True; print(f'gpu{gpu}: out of memory decoding, switching to tiled', flush=True)
    arr = np.nan_to_num(np.asarray(frames)); ok = arr.std() > 0.02
    vid = (np.clip(arr, 0, 1) * 255).round().astype(np.uint8)
    imageio.mimwrite(f"{OUT}/{j['id']}.mp4", list(vid), fps=24, codec='libx264', quality=None, output_params=['-crf', '12', '-pix_fmt', 'yuv420p'])
    print(f"gpu{gpu}: {j['id']} {j['frames']} frames in {time.time() - t0:.0f} s {'OK' if ok else 'BROKEN?'}", flush=True)
    torch.cuda.empty_cache()
print(f'gpu{gpu}: finished', flush=True)

In [ ]:
# 5. Run both GPUs and show progress (this is the long part)
for f in os.listdir(WORK):
    if f.startswith('ready'): os.remove(f'{WORK}/{f}')
procs = [subprocess.Popen(['python', '/tmp/ep02/worker.py', str(g), str(NGPU)], stdout=open(f'{WORK}/log{g}.txt', 'w'), stderr=subprocess.STDOUT) for g in range(NGPU)]
t0 = time.time(); seen = 0
while any(p.poll() is None for p in procs):
    time.sleep(30)
    lines = []
    for g in range(NGPU): lines += [l.strip() for l in open(f'{WORK}/log{g}.txt') if l.startswith('gpu')]
    for l in lines[seen:]: print(f'[{(time.time() - t0) / 60:5.1f} min] {l}', flush=True)
    seen = len(lines)
for g in range(NGPU):
    if procs[g].returncode: print(f'--- gpu{g} stopped with an error; last lines:'); print(''.join(open(f'{WORK}/log{g}.txt').readlines()[-25:]))
print('clips done:', len([f for f in os.listdir(OUT) if f.endswith('.mp4')]), 'of', len(jobs))

In [ ]:
# 6. Pack the clips for download
import zipfile
from IPython.display import FileLink, display
os.chdir('/kaggle/working')
with zipfile.ZipFile('ep02_clips.zip', 'w', zipfile.ZIP_STORED) as z:
    for f in sorted(os.listdir(OUT)):
        if f.endswith('.mp4'): z.write(f'{OUT}/{f}', f)
print(round(os.path.getsize('ep02_clips.zip') / 1e6, 1), 'MB')
display(FileLink('ep02_clips.zip'))